# 第38章：金融机器学习管线

本 Notebook 复算正文第8节：手算标签与可用训练行、三种验证方法在纯噪声特征上的结果、全样本特征选择的泄漏、以及完整构造面板上的预测指标与校准。核心逻辑来自 `quant_course.ml_pipeline`。全部数据为**教学构造**，不是市场证据。

时间约定：signal_time $=D_k$ 收盘；trade_time $=$ label_start $=D_{k+\ell}$；label_end $=$ label_available_at $=D_{k+\ell+h}$。训练行须 label_available_at $\le$ fit_time。

本 Notebook 使用离线教学构造数据，或明确标注的历史结果。新运行输出保存在单独的 `runs/` 子目录；这些内容不构成真实市场研究。


In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch38_ml_pipeline_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

from quant_course import ml_pipeline as mp
ch38 = load_chapter('ch38_ml_pipeline')

## 1. 手算例：3 资产 × 12 日，$\ell=1$，$h=2$

拟合时点 $D_7$：可用 $k\le4$，排除 $k=5,6$ 共 $(\ell+h-1)\times N=6$ 行。

In [ ]:
hand = ch38.hand_example()
lab = pd.DataFrame(hand["labels"]).pivot(index="k", columns="asset", values="y")
display(lab)
print("fit k", hand["fit_time_k"], "used", hand["used_train_dates_k"], "excluded rows", hand["n_excluded_unavailable"])
assert hand["used_train_dates_k"] == [1, 2, 3, 4] and hand["n_excluded_unavailable"] == 6

## 2. 纯噪声特征：随机打乱 K 折、日期块 K 折、日期 walk-forward

随机打乱只用于度量偏差（错误做法），从不用于选模型。约需 20 秒。

In [ ]:
sch = ch38.validation_schemes(seed=3801)
df = pd.DataFrame(sch["results"]).T.drop(columns="n_walk_forward_folds") * 100
display(df.round(2))
assert df.loc["knn", "shuffled_kfold_r2_vs_zero"] > 40 and df.loc["knn", "walk_forward_r2_vs_zero"] < 0

## 3. 全样本特征选择 vs. Pipeline 内选择（200 个纯噪声特征选 10 个）

In [ ]:
leak = ch38.selection_leak(seed=3802)
display(pd.DataFrame(leak["results"]).T.round(5))
assert leak["results"]["full_sample_selection"]["r2_vs_zero"] > leak["results"]["train_only_selection"]["r2_vs_zero"]

## 4. 完整构造面板：Ridge 日期 walk-forward

分开看水平与截面：池化 $R^2$ 可以为负，而截面 $R^2$ 与 IC 为正。

In [ ]:
pc = ch38.panel_case(seed=3803)
m = pc["metrics"]
print({k: round(m[k], 5) for k in ("oos_r2_vs_zero", "oos_r2_vs_train_mean", "ic_mean", "ic_t", "hit_rate")})
print("cross-sectional R2", round(pc["cross_sectional_r2_demeaned"], 5), "slopes", round(pc["calibration_slope_y_on_pred"], 3),
      round(pc["cross_sectional_calibration_slope"], 3), "alphas", pc["chosen_alpha_by_fold"])
assert pc["excluded_rows_per_fold"] == [200]
dec = np.array(pc["within_date_decile_mean_demeaned_y"]) * 100
plt.bar(range(1, 11), dec); plt.axhline(0, color="grey", lw=0.6)
plt.xlabel("within-date prediction decile"); plt.ylabel("mean demeaned 5-day return (%)"); plt.title("CONSTRUCTED"); plt.savefig(RUN_DIR / "ch38_ml_pipeline_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()